<a href="https://colab.research.google.com/github/uniqueeee777/stability-aware-multiview-food-security/blob/main/02_Preprocessing_and_MultiView_Representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load data dan verifikasi reproduksibilitas

In [ ]:

# ============================================================
# CELL 1 — REPRODUCIBLE DATA LOADING
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display
from scipy.spatial.distance import pdist, squareform
from scipy.stats import spearmanr

import pandas as pd
import numpy as np
import hashlib
import json

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

INPUT = (
    ROOT / "01_data" / "01_raw"
    / "MATRIX_9VAR_CLUSTERING_85KAB_2025.csv"
)

EXPECTED_SHA256 = (
    "ce932d322eb49cbdb6ea3b0a8768ca89c2e9"
    "e7f0722232883541d891089ca6fa"
)

assert INPUT.exists(), f"File tidak ditemukan: {INPUT}"

actual_hash = hashlib.sha256(
    INPUT.read_bytes()
).hexdigest()

assert actual_hash == EXPECTED_SHA256, (
    "Isi file berbeda dari dataset yang telah diaudit."
)

ID_COLS = [
    "kode_bps", "kode_gaul2",
    "provinsi", "kabupaten_kota"
]

VIEWS = {
    "Food": [
        "F1_ikp_ketersediaan",
        "F2_ikp_keterjangkauan",
        "F3_ikp_pemanfaatan"
    ],
    "Climate": [
        "C1_anomali_hujan_mm",
        "C2_cdd_hari",
        "C3_hari_hujan_ekstrem"
    ],
    "Accessibility": [
        "A1_persen_tanpa_angkutan",
        "A2_persen_jalan_tanah_lainnya",
        "A3_persen_tanpa_pertokoan_pasar"
    ]
}

FEATURES = sum(VIEWS.values(), [])

df = pd.read_csv(
    INPUT,
    dtype={
        "kode_bps": "string",
        "kode_gaul2": "string"
    }
)

assert all(c in df.columns for c in ID_COLS + FEATURES)
assert df.shape == (85, 13)
assert df["kode_bps"].is_unique

X = df[FEATURES].apply(
    pd.to_numeric, errors="raise"
)

assert X.notna().all().all()
assert np.isfinite(X.to_numpy()).all()

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_UTC"
)

OUT = (
    ROOT / "03_outputs"
    / "02_representation" / RUN_ID
)

OUT.mkdir(parents=True, exist_ok=True)

print("DATA VERIFIED")
print("Dimensi:", X.shape)
print("Hash:", actual_hash)
print("Output:", OUT)


Mounted at /content/drive
DATA VERIFIED
Dimensi: (85, 9)
Hash: ce932d322eb49cbdb6ea3b0a8768ca89c2e9e7f0722232883541d891089ca6fa
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/02_representation/20261018_173812_UTC


# Transformasi dan konstruksi matriks jarak

In [ ]:

# ============================================================
# CELL 2 — REPRESENTATION & DISTANCE CONSTRUCTION
# ============================================================

RECIPES = {
    "R0_raw_zscore": [],
    "R1_logCDD_zscore": ["C2_cdd_hari"],
    "R2_logCDD_A1_A2_zscore": [
        "C2_cdd_hari",
        "A1_persen_tanpa_angkutan",
        "A2_persen_jalan_tanah_lainnya"
    ],
    "R3_raw_robust": []
}

WEIGHTS = {
    "Food": 1 / 3,
    "Climate": 1 / 3,
    "Accessibility": 1 / 3
}

assert np.isclose(sum(WEIGHTS.values()), 1)


def fit_representation(data, recipe):
    """
    Fit preprocessing pada data yang diberikan.

    Pada eksperimen subsampling nanti, fungsi ini harus
    di-fit ulang pada subset, bukan pada data lengkap.
    """

    Y = data.astype(float).copy()

    for col in RECIPES[recipe]:
        if (Y[col] < 0).any():
            raise ValueError(
                f"log1p tidak valid untuk {col}"
            )
        Y[col] = np.log1p(Y[col])

    if recipe == "R3_raw_robust":
        center = Y.median()
        scale = (
            Y.quantile(0.75) - Y.quantile(0.25)
        )
    else:
        center = Y.mean()
        scale = Y.std(ddof=1)

    if (scale <= 0).any() or scale.isna().any():
        raise ValueError(
            "Skala nol atau tidak valid."
        )

    Z = (Y - center) / scale

    return Z, {
        "center": center.to_dict(),
        "scale": scale.to_dict()
    }


def construct_distances(Z):
    """
    View-specific Euclidean distances.

    Setiap view dinormalisasi agar RMS jarak
    antarpasangan = sqrt(2).

    Matriks gabungan diverifikasi terhadap
    Euclidean distance pada fitur berbobot.
    """

    view_D = {}
    blocks = []
    view_factors = {}

    for view, cols in VIEWS.items():

        A = Z[cols].to_numpy(dtype=float)
        p = len(cols)

        D = squareform(
            pdist(A, metric="euclidean")
        ) / np.sqrt(p)

        upper = D[np.triu_indices(len(D), k=1)]
        rms = np.sqrt(np.mean(upper ** 2))

        if not np.isfinite(rms) or rms <= 0:
            raise ValueError(
                f"Jarak view tidak valid: {view}"
            )

        factor = np.sqrt(2) / rms

        view_D[view] = D * factor
        view_factors[view] = float(factor)

        blocks.append(
            A * factor * np.sqrt(WEIGHTS[view] / p)
        )

    Phi = np.concatenate(blocks, axis=1)

    D_fused = squareform(
        pdist(Phi, metric="euclidean")
    )

    D_check = np.sqrt(
        sum(
            WEIGHTS[v] * view_D[v] ** 2
            for v in VIEWS
        )
    )

    assert np.allclose(D_fused, D_check)
    assert np.allclose(D_fused, D_fused.T)
    assert np.allclose(np.diag(D_fused), 0)
    assert np.isfinite(D_fused).all()

    return Phi, D_fused, view_D, view_factors


REPRESENTATIONS = {}

for recipe in RECIPES:

    Z, fit_parameters = fit_representation(
        X, recipe
    )

    Phi, D_fused, D_views, factors = (
        construct_distances(Z)
    )

    REPRESENTATIONS[recipe] = {
        "Z": Z,
        "Phi": Phi,
        "D": D_fused,
        "D_views": D_views,
        "fit_parameters": fit_parameters,
        "view_factors": factors
    }

    # Simpan representasi dengan identitas wilayah
    output_data = pd.concat(
        [
            df[ID_COLS].reset_index(drop=True),
            Z.reset_index(drop=True)
        ],
        axis=1
    )

    output_data.to_csv(
        OUT / f"{recipe}_features.csv",
        index=False
    )

    np.savez_compressed(
        OUT / f"{recipe}_distances.npz",
        kode_bps=df["kode_bps"].astype(str).to_numpy(),
        fused=D_fused,
        food=D_views["Food"],
        climate=D_views["Climate"],
        accessibility=D_views["Accessibility"],
        euclidean_features=Phi
    )

print("ALL REPRESENTATIONS CONSTRUCTED")
print("Jumlah alternatif:", len(REPRESENTATIONS))


ALL REPRESENTATIONS CONSTRUCTED
Jumlah alternatif: 4


# Sensitivitas representasi sebelum pemodelan

In [ ]:

# ============================================================
# CELL 3 — REPRESENTATION SENSITIVITY
# ============================================================

BASE = "R0_raw_zscore"

D_ref = REPRESENTATIONS[BASE]["D"]
n = len(df)


def mean_knn_overlap(D1, D2, k=5):

    A = D1.copy()
    B = D2.copy()

    np.fill_diagonal(A, np.inf)
    np.fill_diagonal(B, np.inf)

    neighbors_a = np.argsort(
        A, axis=1, kind="stable"
    )[:, :k]

    neighbors_b = np.argsort(
        B, axis=1, kind="stable"
    )[:, :k]

    overlaps = [
        len(
            set(neighbors_a[i])
            & set(neighbors_b[i])
        ) / k
        for i in range(len(A))
    ]

    return float(np.mean(overlaps))


records = []

for name, result in REPRESENTATIONS.items():

    D = result["D"]

    rho = spearmanr(
        squareform(D_ref),
        squareform(D)
    ).statistic

    overlap = mean_knn_overlap(
        D_ref, D, k=5
    )

    records.append({
        "representation": name,
        "distance_spearman": float(rho),
        "knn5_overlap": float(overlap),
        "min_distance": float(
            squareform(D).min()
        ),
        "max_distance": float(
            squareform(D).max()
        )
    })

sensitivity = pd.DataFrame(records)

display(sensitivity.round(4))

sensitivity.to_csv(
    OUT / "representation_sensitivity.csv",
    index=False
)

# Metadata reproduksibilitas
manifest = {
    "run_id": RUN_ID,
    "input_sha256": actual_hash,
    "n_regions": n,
    "feature_order": FEATURES,
    "views": VIEWS,
    "recipes": RECIPES,
    "weights": WEIGHTS,
    "status": "PREPROCESSING_DIAGNOSTIC_ONLY",
    "fit_parameters": {
        r: REPRESENTATIONS[r]["fit_parameters"]
        for r in RECIPES
    },
    "view_factors": {
        r: REPRESENTATIONS[r]["view_factors"]
        for r in RECIPES
    }
}

(OUT / "representation_manifest.json").write_text(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("CELL 3 COMPLETED")
print("Saved to:", OUT)


,representation,distance_spearman,knn5_overlap,min_distance,max_distance
0,R0_raw_zscore,1.0000,1.0000,0.1921,3.2470
1,R1_logCDD_zscore,0.9871,0.8941,0.2099,3.2532
2,R2_logCDD_A1_A2_zscore,0.9420,0.7012,0.2221,3.1715
3,R3_raw_robust,0.9737,0.8659,0.1912,3.2069


CELL 3 COMPLETED
Saved to: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/02_representation/20261018_173812_UTC


# Konstruksi graf kemiripan dan pemeriksaan matematis

In [ ]:

# ============================================================
# CELL 4 — GRAPH CONSTRUCTION AND SPECTRAL DIAGNOSTICS
# References:
# Zelnik-Manor & Perona (2004): local scaling
# von Luxburg (2007): graph Laplacian
# ============================================================

from scipy.sparse.csgraph import connected_components
import numpy as np
import pandas as pd
import json

GRAPH_K_GRID = [5, 8, 12, 16, 24]
LOCAL_SCALE_K = 7

assert set(WEIGHTS) == set(VIEWS)
assert np.isclose(sum(WEIGHTS.values()), 1.0)


def build_affinity(D, k_graph, k_scale):
    """
    Symmetric union-kNN graph with local Gaussian scaling.

    W_ij = exp[-d_ij^2 / (sigma_i * sigma_j)]
    for edges in the symmetric union-kNN graph.

    sigma_i = distance to the k_scale-th nearest neighbor.
    """

    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    if D.shape != (n, n):
        raise ValueError("Distance matrix must be square.")

    if not np.allclose(D, D.T, atol=1e-10):
        raise ValueError("Distance matrix is not symmetric.")

    if not np.allclose(np.diag(D), 0, atol=1e-10):
        raise ValueError("Distance diagonal must be zero.")

    if not np.isfinite(D).all() or (D < 0).any():
        raise ValueError("Invalid distances detected.")

    if not (1 <= k_graph < n and 1 <= k_scale < n):
        raise ValueError("Invalid neighborhood parameters.")

    # Exclude self-neighbors
    D_work = D.copy()
    np.fill_diagonal(D_work, np.inf)

    order = np.argsort(
        D_work,
        axis=1,
        kind="stable"
    )

    sigma = np.take_along_axis(
        D_work,
        order[:, [k_scale - 1]],
        axis=1
    ).ravel()

    if not np.isfinite(sigma).all() or (sigma <= 0).any():
        raise ValueError(
            "Invalid local bandwidth. Review duplicate points."
        )

    # Directed neighborhood mask
    neighbors = order[:, :k_graph]

    adjacency = np.zeros(
        (n, n), dtype=bool
    )

    adjacency[
        np.arange(n)[:, None],
        neighbors
    ] = True

    # Symmetric union, not mutual kNN
    adjacency = adjacency | adjacency.T

    W = np.exp(
        -(D ** 2) /
        (sigma[:, None] * sigma[None, :])
    )

    W = np.where(adjacency, W, 0.0)
    np.fill_diagonal(W, 0.0)

    if not np.allclose(W, W.T, atol=1e-12):
        raise AssertionError("Affinity is not symmetric.")

    if not np.isfinite(W).all() or (W < 0).any():
        raise ValueError("Invalid affinity weights.")

    return W


def graph_diagnostics(W):
    """
    Graph topology and normalized Laplacian diagnostics.
    Does not perform clustering.
    """

    n = W.shape[0]

    degree = W.sum(axis=1)

    if np.any(degree <= 0):
        raise ValueError("Graph contains isolated nodes.")

    n_components, component_labels = (
        connected_components(
            (W > 0).astype(np.uint8),
            directed=False
        )
    )

    inv_sqrt_degree = 1.0 / np.sqrt(degree)

    L_sym = (
        np.eye(n)
        - inv_sqrt_degree[:, None]
        * W
        * inv_sqrt_degree[None, :]
    )

    eigenvalues = np.linalg.eigvalsh(L_sym)

    component_sizes = np.bincount(component_labels)

    edges = int(
        np.count_nonzero(
            np.triu(W, k=1)
        )
    )

    return {
        "components": int(n_components),
        "smallest_component": int(component_sizes.min()),
        "edges": edges,
        "edge_density": edges / (n * (n - 1) / 2),
        "degree_min": float(degree.min()),
        "degree_median": float(np.median(degree)),
        "lambda_2": float(eigenvalues[1]),
        "zero_eigenvalues": int(
            (np.abs(eigenvalues) < 1e-8).sum()
        )
    }


# Evaluate all pre-specified preprocessing recipes
graph_records = []

for recipe, result in REPRESENTATIONS.items():

    for k_graph in GRAPH_K_GRID:

        view_graphs = {}

        for view in VIEWS:

            W = build_affinity(
                result["D_views"][view],
                k_graph=k_graph,
                k_scale=LOCAL_SCALE_K
            )

            view_graphs[view] = W

        # Equal Frobenius normalization before fusion
        W_fused = np.zeros_like(
            next(iter(view_graphs.values()))
        )

        for view, W in view_graphs.items():

            norm = np.linalg.norm(
                W, ord="fro"
            )

            if not np.isfinite(norm) or norm <= 0:
                raise ValueError(
                    f"Invalid graph norm: {view}"
                )

            W_fused += (
                WEIGHTS[view] * W / norm
            )

        assert np.allclose(W_fused, W_fused.T)
        assert np.allclose(np.diag(W_fused), 0)

        # Assess individual and fused graphs
        for graph_name, W in {
            **view_graphs,
            "FUSED": W_fused
        }.items():

            stats = graph_diagnostics(W)

            graph_records.append({
                "representation": recipe,
                "k_graph": k_graph,
                "k_scale": LOCAL_SCALE_K,
                "graph": graph_name,
                **stats
            })

graph_qc = pd.DataFrame(graph_records)

graph_qc.to_csv(
    OUT / "graph_diagnostics.csv",
    index=False
)

graph_protocol = {
    "graph_type": "symmetric_union_kNN",
    "kernel": "local_scaling_gaussian",
    "local_scale_neighbor": LOCAL_SCALE_K,
    "graph_k_candidates": GRAPH_K_GRID,
    "fusion_normalization": "frobenius",
    "fusion_weights": WEIGHTS,
    "status": "DIAGNOSTIC_NOT_MODEL_SELECTION"
}

(OUT / "graph_protocol.json").write_text(
    json.dumps(
        graph_protocol,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("GRAPH CONSTRUCTION COMPLETED")
print("Configurations evaluated:", len(graph_qc))
print("Saved:", OUT / "graph_diagnostics.csv")


GRAPH CONSTRUCTION COMPLETED
Configurations evaluated: 80
Saved: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/02_representation/20261018_173812_UTC/graph_diagnostics.csv


In [ ]:

# ============================================================
# CELL 5 — GRAPH DIAGNOSTIC SUMMARY
# ============================================================

# 1. Graph baseline R0
baseline_graphs = graph_qc.loc[
    graph_qc["representation"] == "R0_raw_zscore",
    [
        "graph", "k_graph", "components",
        "smallest_component", "edge_density",
        "degree_min", "lambda_2",
        "zero_eigenvalues"
    ]
].copy()

print("BASELINE R0 GRAPH DIAGNOSTICS:")
display(
    baseline_graphs.sort_values(
        ["graph", "k_graph"]
    ).round(5)
)

# 2. Fused graph comparison across representations
fused_summary = graph_qc.loc[
    graph_qc["graph"] == "FUSED",
    [
        "representation", "k_graph",
        "components", "edge_density",
        "lambda_2"
    ]
].copy()

print("\nMULTI-VIEW FUSED GRAPH:")
display(
    fused_summary.sort_values(
        ["representation", "k_graph"]
    ).round(5)
)

# 3. Flag any disconnected graphs
disconnected = graph_qc.loc[
    graph_qc["components"] > 1
].copy()

print("\nDisconnected graph configurations:",
      len(disconnected))

if not disconnected.empty:
    display(disconnected[
        [
            "representation", "graph",
            "k_graph", "components",
            "smallest_component"
        ]
    ])

# 4. Check spectral/topological consistency
inconsistent = graph_qc.loc[
    graph_qc["components"]
    != graph_qc["zero_eigenvalues"]
]

if not inconsistent.empty:
    print(
        "WARNING: Numerical eigenvalue/component "
        "consistency requires inspection."
    )
    display(inconsistent)
else:
    print(
        "Graph components and zero-eigenvalue "
        "counts are consistent."
    )

print("\nCELL 5 COMPLETED")


BASELINE R0 GRAPH DIAGNOSTICS:


,graph,k_graph,components,smallest_component,edge_density,degree_min,lambda_2,zero_eigenvalues
2,Accessibility,5,1,85,0.07871,1.41351,0.00895,1
6,Accessibility,8,1,85,0.12549,1.91234,0.02989,1
10,Accessibility,12,1,85,0.19048,2.12874,0.04295,1
14,Accessibility,16,1,85,0.25434,2.42333,0.04992,1
18,Accessibility,24,1,85,0.37787,2.75273,0.06195,1
1,Climate,5,1,85,0.07703,1.79014,0.00914,1
5,Climate,8,1,85,0.12045,2.12695,0.01329,1
9,Climate,12,1,85,0.17927,2.22577,0.01840,1
13,Climate,16,1,85,0.23557,2.30654,0.02170,1
17,Climate,24,1,85,0.34566,2.31238,0.02849,1



MULTI-VIEW FUSED GRAPH:


,representation,k_graph,components,edge_density,lambda_2
3,R0_raw_zscore,5,1,0.20224,0.23948
7,R0_raw_zscore,8,1,0.29664,0.27081
11,R0_raw_zscore,12,1,0.41709,0.29246
15,R0_raw_zscore,16,1,0.52465,0.31065
19,R0_raw_zscore,24,1,0.69748,0.32889
23,R1_logCDD_zscore,5,1,0.20392,0.23140
27,R1_logCDD_zscore,8,1,0.29860,0.26494
31,R1_logCDD_zscore,12,1,0.41373,0.28748
35,R1_logCDD_zscore,16,1,0.51905,0.30178
39,R1_logCDD_zscore,24,1,0.69272,0.32069



Disconnected graph configurations: 0
Graph components and zero-eigenvalue counts are consistent.

CELL 5 COMPLETED
